# Bước 1 - Phân rã Context thành Candidate Evidence Sentences
## Module 7.1: Vietnamese Sentence Splitting với Kỹ thuật Masking Protection

---

### Mục tiêu:
Bài báo gốc (Context) là một đoạn văn dài gồm nhiều câu. Để phục vụ việc tìm kiếm bằng chứng cho từng câu Tuyên bố (Claim), bước đầu tiên là tách bài báo thành các câu ứng viên (**Candidate Evidence Pool**).
Thuật toán tách câu được bảo vệ bởi **Masking Protection**:
- Không tách nhầm ở các số thập phân (ví dụ `8.5%`, `10.000 m3`).
- Không tách nhầm ở các từ viết tắt phổ biến (`TP.HCM`, `TS.`, `PGS.TS.`, `VNĐ.`, `Q.1`).

In [1]:
import os
import re
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("../..").resolve()
DATA_DEV_PATH = PROJECT_ROOT / "data/processed/common_cleaned/vifactcheck_dev_common_cleaned.csv"
OUTPUT_DIR = PROJECT_ROOT / "data/processed/retrieval"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df_dev = pd.read_csv(DATA_DEV_PATH)
print(f"✓ Đã nạp thành công tập Dev: {len(df_dev):,} bài báo")
display(df_dev[["index", "Statement", "labels", "Context", "Evidence"]].head(3))

✓ Đã nạp thành công tập Dev: 723 bài báo


,index,Statement,labels,Context,Evidence
0,6040,"Vào tháng 4.1930 TL Nhà vua Na Uy Harald V, Vu...",1,"Saigon Morin, khách sạn 4 sao hàng đầu tại Huế...",Vua hề Charlie Chaplin (vua hề Sác lô) và vợ t...
1,5599,Nhiều chi bộ chỉ mua báo đảng mà không quan tâ...,1,"(Chinhphu.vn) - Bí thư Trung ương Đảng, Trưởng...","Việc mua, đọc, sử dụng báo, tạp chí của Đảng t..."
2,1947,"Công ty TNHH Mua bán nợ DSP, địa chỉ 91 Pasteu...",1,(NLĐO)- Sau khi mua khoản nợ từ Công ty Mirae ...,"Công ty TNHH Mua bán nợ DSP, địa chỉ: Tầng 4, ..."


In [2]:
ABBREVIATIONS = [
    r"TP\.HCM", r"T\.P\.HCM", r"Hà Nội", r"PGS\.TS", r"GS\.TS", r"TS\.", r"ThS\.", r"BS\.",
    r"Ths\.", r"Bs\.", r"ông\.", r"bà\.", r"Đ\/c\.", r"đ\/c\.", r"P\.", r"Q\.", r"TX\.",
    r"TT\.", r"H\.", r"NXB\.", r"VNĐ\.", r"VND\.", r"USD\.", r"EUR\."
]

def split_vietnamese_sentences(text: str) -> list[str]:
    """Tách đoạn văn tiếng Việt thành danh sách câu với bảo vệ viết tắt và số thực."""
    if not isinstance(text, str) or not text.strip():
        return []
    
    masked = text.strip()
    abbrev_map = {}
    for idx, pattern in enumerate(ABBREVIATIONS):
        key = f"__ABBR_{idx}__"
        matches = re.findall(pattern, masked, flags=re.IGNORECASE)
        for m in set(matches):
            masked = masked.replace(m, key)
            abbrev_map[key] = m

    num_map = {}
    num_matches = re.findall(r"\b\d+[\.,]\d+\b", masked)
    for idx, m in enumerate(set(num_matches)):
        key = f"__NUM_{idx}__"
        masked = masked.replace(m, key)
        num_map[key] = m

    raw_sents = re.split(r"(?<=[.?!…\n])\s+", masked)
    
    clean_sents = []
    for s in raw_sents:
        for k, v in num_map.items():
            s = s.replace(k, v)
        for k, v in abbrev_map.items():
            s = s.replace(k, v)
        s = s.strip()
        if len(s) > 10:
            clean_sents.append(s)
            
    return clean_sents

# Thử nghiệm hàm tách câu
demo_context = "PGS.TS. Trần Đắc Phu cho biết: Tỷ lệ tiêm chủng đạt 95.8% tại TP.HCM trong quý 1 năm 2024. Đây là kết quả tích cực!"
print("Demo tách câu:")
for i, s in enumerate(split_vietnamese_sentences(demo_context), 1):
    print(f"  Câu {i}: {s}")

Demo tách câu:
  Câu 1: Trần Đắc Phu cho biết: Tỷ lệ tiêm chủng đạt 95.8% tại TP.HCM trong quý 1 năm 2024.
  Câu 2: Đây là kết quả tích cực!


In [3]:
candidates_list = []

for _, row in df_dev.iterrows():
    claim_id = f"dev_{row['index']}"
    context = str(row["Context"]) if pd.notna(row["Context"]) else ""
    sentences = split_vietnamese_sentences(context)
    
    for s_idx, sent in enumerate(sentences):
        candidates_list.append({
            "claim_id": claim_id,
            "claim_index": row["index"],
            "sentence_id": f"{claim_id}_s{s_idx:03d}",
            "sentence_text": sent
        })

df_candidates = pd.DataFrame(candidates_list)
output_path = OUTPUT_DIR / "evidence_candidates.csv"
df_candidates.to_csv(output_path, index=False)

print(f"✓ Tổng số câu ứng viên được tạo: {len(df_candidates):,}")
print(f"✓ Trung bình mỗi bài báo phân rã thành: {len(df_candidates) / len(df_dev):.1f} câu")
print(f"✓ Đã lưu file: {output_path}")
display(df_candidates.head(6))

✓ Tổng số câu ứng viên được tạo: 12,823
✓ Trung bình mỗi bài báo phân rã thành: 17.7 câu
✓ Đã lưu file: /Users/mivu/Documents/Artificial Intelligence/CS221-NLP/Project-NLP-Fact-Checking/notebooks/Pipeline IR & IE_IR_IE_Reranking/outputs/evidence_candidates.csv


,claim_id,claim_index,sentence_id,sentence_text
0,dev_6040,6040,dev_6040_s000,"Saigon Morin, khách sạn 4 sao hàng đầu tại Huế..."
1,dev_6040,6040,dev_6040_s001,Khách sạn có 4 mặt tiền thuộc các giao lộ Lê L...
2,dev_6040,6040,dev_6040_s002,"Ngoài việc phục vụ du khách, Saigon Morin còn ..."
3,dev_6040,6040,dev_6040_s003,Khách sạn nhanh chóng trở thành trung tâm thươ...
4,dev_6040,6040,dev_6040_s004,Hình ảnh khách sạn những ngày đầu mới thành lậ...
5,dev_6040,6040,dev_6040_s005,"Đến năm 1907, khách sạn đã được chuyển nhượng ..."
